# Notebook 03 — Modelo, Entrenamiento y Experimentación
## Detección de Muelas del Juicio Impactadas con YOLOv8

**Materia:** Redes Neuronales  
**Docente:** Ing. Pablo Marinozi  
**Repositorio:** https://github.com/JulianOliveraBalls/wisdomscan

---

Este notebook documenta la experimentación completa: desde la primera prueba con YOLOv8s hasta el modelo de producción. Todos los experimentos están documentados con sus resultados hardcodeados — no se re-entrenan al correr el notebook.

| Sección | Contenido |
|---------|----------|
| **a** | Arquitectura elegida y estrategia de fine-tuning |
| **b** | Configuración del entrenamiento |
| **c** | Experimentación completa — 4 fases, 18 experimentos |
| **d** | Evaluación del modelo final — métricas, curvas, análisis de errores |


---

## Sección 0 — Setup


In [ ]:
import subprocess, sys, os, json, random, warnings, shutil
from pathlib import Path
from collections import defaultdict
warnings.filterwarnings('ignore')

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *pkgs, '-q'])

try:
    import ultralytics
except ImportError:
    pip_install('ultralytics')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import torch
from ultralytics import YOLO

def log(msg, level='INFO'):
    icons = {'INFO':'[INFO]','OK':'[OK]  ','WARN':'[WARN]','ERR':'[ERR] ','DATA':'[DATA]'}
    print(f'{icons.get(level,"[INFO]")} {msg}')

try:
    import google.colab
    IN_COLAB = os.path.exists('/content')
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive; drive.mount('/drive')
    REPO_ROOT = Path('/content/wisdomscan')
    if not REPO_ROOT.exists():
        subprocess.run(['git','clone',
            'https://github.com/JulianOliveraBalls/wisdomscan.git',
            str(REPO_ROOT)], check=True)
    DRIVE_DIR = Path('/drive/MyDrive/dentex_runs')
else:
    _nb = Path(globals().get('__vsc_ipynb_file__',
               globals().get('__file__', str(Path.cwd()/'dev'/'nb.ipynb'))))
    REPO_ROOT = _nb.parent.parent
    DRIVE_DIR = REPO_ROOT / 'dev'

PROCESSED   = REPO_ROOT / 'data' / 'processed'
OUTPUTS_DIR = REPO_ROOT / 'data' / 'outputs'
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

YOLO_DIR    = PROCESSED / 'yolo_dataset'
YOLO_QUAD   = PROCESSED / 'yolo_quad_flip'
YOLO_MERGED = PROCESSED / 'yolo_merged'
YOLO_G8b    = PROCESSED / 'yolo_g8b_small'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
log(f'device: {device}  |  IN_COLAB: {IN_COLAB}', 'OK')


---

## a) Modelo preentrenado y estrategia de fine-tuning

### Modelo elegido: YOLOv8

Se eligió **YOLOv8** de Ultralytics como arquitectura base. YOLOv8 es un detector de objetos single-stage que realiza en un único forward pass tanto la localización (bounding box) como la clasificación de los objetos.

**Justificación frente a alternativas:**

| Alternativa | Por qué se descartó |
|-------------|---------------------|
| ResNet/EfficientNet + clasificador | Son arquitecturas de clasificación — no localizan. Requerirían un pipeline separado de sliding window o region proposals. |
| ViT (Vision Transformer) | Diseñado para clasificación global. Adaptarlo a detección requiere DETR o similar, con mayor complejidad y menor madurez de tooling. |
| Faster R-CNN | Two-stage detector: más preciso pero 5–10x más lento que YOLO en inferencia. El caso de uso (app web) prioriza velocidad. |
| **YOLOv8** | **Single-stage, estado del arte en velocidad/precisión, API simple, soporte nativo de augmentation sincronizada con boxes, exportación ONNX/TensorRT. Ecosistema maduro.** |

### Arquitectura de YOLOv8m

El proyecto usó dos variantes a lo largo de los experimentos:

| Variante | Parámetros | Uso |
|----------|-----------|-----|
| **YOLOv8s** (small) | ~11M | Fases 1–3 (exploración rápida, menor costo computacional) |
| **YOLOv8m** (medium) | ~26M | Fases G y final (mayor capacidad, mejores métricas absolutas) |


### Modificaciones a la arquitectura

YOLOv8 reemplaza automáticamente la capa de clasificación final al especificar `nc` (number of classes) en el archivo `dataset.yaml`. No se agregaron capas adicionales.

| Fase | nc | Clases |
|------|----|--------|
| Fases 1–3, G1–G7 | 2 | `erupted`, `impacted` |
| G8, G8b (final) | 1 | `impacted` (solo) |

### Estrategia de fine-tuning

Se aplicó **fine-tuning completo end-to-end** en todos los experimentos — ninguna capa del backbone se congeló en los experimentos principales.

**Justificación:**

- El dominio fuente (COCO — fotos de objetos naturales) es muy diferente del dominio objetivo (radiografías médicas en escala de grises). Las features de bajo nivel (bordes, texturas) no son directamente transferibles — el backbone necesita adaptarse.
- El dataset es pequeño (~300–7800 imágenes según la fase) pero suficiente para fine-tuning completo con learning rate bajo y augmentation agresiva.
- Se probó congelar el backbone en Exp12 — el resultado fue peor (mAP50 0.748 vs 0.769 sin congelamiento), confirmando que el fine-tuning completo es la estrategia correcta.

**Excepción — Exp9 (pipeline jerárquico):** se usó transfer learning progresivo en 3 etapas (cuadrantes → FDI → erupted/impacted), inicializando cada etapa con los pesos de la anterior. Esta es la única excepción al fine-tuning directo end-to-end.

**Backbone dental (Exp10–12):** se usó el modelo [nsitnov/8024-yolov8-model](https://huggingface.co/nsitnov/8024-yolov8-model) como punto de partida en lugar de COCO. Preentrenado en 8 patologías dentales, sus features son más cercanas al dominio objetivo.


---

## b) Configuración del entrenamiento

### Función de pérdida

YOLOv8 usa una pérdida compuesta internamente:

| Componente | Función | Pondera |
|-----------|---------|--------|
| Localización | CIoU Loss (Complete IoU) | Error en posición, tamaño y relación de aspecto del box |
| Clasificación | Binary Cross-Entropy | Error de clase por objeto |
| Objectness | Binary Cross-Entropy | Confianza de que hay un objeto en la celda |

No se usó Focal Loss ya que el desbalance erupted/impacted (~37/63%) es moderado. Se probó penalizar erupted con `cls=1.5` en Exp_G4 — no mejoró las métricas en test.

### Optimizador y learning rate

| Parámetro | Valor | Justificación |
|-----------|-------|---------------|
| Optimizador | **AdamW** | Mejor convergencia que SGD en dominios con poco data |
| `lr0` | 0.001–0.0001 | Bajo para preservar features preentrenadas |
| `lrf` | 0.01 | Factor final del scheduler coseno |
| `momentum` | 0.937 | Default YOLOv8 |
| `weight_decay` | 0.0005 | Regularización L2 |
| `warmup_epochs` | 3 | Rampa suave al inicio para no dañar pesos preentrenados |

### Scheduler

YOLOv8 usa un **cosine annealing** del learning rate desde `lr0` hasta `lr0 × lrf` a lo largo de todas las épocas. No se usó ReduceLROnPlateau — el coseno es más estable con datasets pequeños donde el plateau puede ser ruidoso.

### Configuraciones por fase

| Fase | Modelo | Épocas | Batch | lr0 | Aug principal | Dataset train |
|------|--------|--------|-------|-----|---------------|---------------|
| 1 (Exp1–7) | YOLOv8s | 10 | 8 | 0.01 | variable | 308–1744 |
| 2 (top 3) | YOLOv8s | 30 | 8 | 0.01 | fliplr=0.5 | 308–1744 |
| 3 (Exp9–12) | YOLOv8s | 30–50 | 8 | 0.01 | fliplr=0.5 | 308–1744 |
| G (G1–G6) | YOLOv8m | 30–60 | 8 | 0.001 | fliplr, hsv | 2619–6630 |
| G7–G8b | YOLOv8m | 25–60 | 8 | 0.0001 | fliplr, hsv | 127–19890 |

### Hardware y tiempos

| Recurso | Detalle |
|---------|--------|
| Plataforma | Google Colab |
| GPU | NVIDIA Tesla T4 (16 GB VRAM) |
| Tiempo por experimento | ~8–15 min (10 epochs) / ~25–40 min (30 epochs) / ~60–90 min (60 epochs) |
| Tiempo total acumulado | ~12–15 horas de GPU (todas las fases) |
| Checkpoints | `best.pt` y `last.pt` guardados en Google Drive cada 15 min (daemon thread) |


---

## c) Experimentación

La experimentación se organizó en **4 fases** con preguntas de investigación concretas. Todos los resultados son hardcodeados a partir de las ejecuciones reales en Colab.


### Fase 1 — Exploración rápida de dataset y augmentation (10 epochs)

**Pregunta:** ¿Qué combinación de dataset + augmentation da el mejor punto de partida?

**Modelo base:** YOLOv8s pretrained en COCO. Fine-tuning completo end-to-end.

| Exp | Dataset | N train | Augmentation YOLO | mAP50 val | mAP50-95 val |
|-----|---------|---------|-------------------|-----------|--------------|
| Exp1 | yolo_dataset (base) | 308 | ninguna | 0.686 | 0.450 |
| Exp2 | yolo_dataset | 308 | fliplr=0.5 | 0.684 | 0.443 |
| Exp3 | yolo_dataset | 308 | full aug | 0.714 | 0.446 |
| Exp4 | yolo_dataset | 308 | mixaug (scale+shear) | 0.732 | 0.454 |
| Exp5 | yolo_quad (cuadrantes) | 1026 | ninguna | 0.696 | 0.464 |
| **Exp6** | **yolo_quad_flip** | **1744** | **fliplr=0.5** | **0.739** | **0.491** |
| Exp7 | yolo_quad_flip | 1744 | full aug | 0.666 | 0.424 |


**Hallazgos:**
- Exp6 gana: cuadrantes + flip horizontal es la mejor combinación a 10 epochs
- Exp7 (misma expansión + aug agresiva) queda último: la augmentation fuerte con pocos epochs desestabiliza el entrenamiento — el modelo no tiene tiempo de converger
- La expansión de cuadrantes (×3–5 imágenes) supera al dataset base incluso sin aug
- El flip horizontal tiene sentido clínico: las muelas aparecen simétricamente en ambos lados


### Fase 2 — Refinamiento del top 3 (30 epochs)

**Pregunta:** ¿Cuánto mejoran los mejores configs de Fase 1 con más epochs?

Top 3 seleccionados de Fase 1: Exp6, Exp3, Exp5.

| Exp | mAP50 (10 ep) | mAP50 (30 ep) | Delta |
|-----|--------------|--------------|-------|
| Exp6 — quad+flip | 0.739 | **0.785** | +0.046 |
| Exp3 — base full aug | 0.714 | 0.764 | +0.050 |
| Exp5 — quad sin aug | 0.696 | 0.714 | +0.018 |


**Hallazgos:**
- Exp6 sigue siendo el mejor pero la brecha con Exp3 se acorta a 30 epochs
- Exp5 (sin augmentation extra) mejora poco — confirma que augmentation es necesaria
- La tendencia es clara: más epochs siempre ayuda, pero el rendimiento marginal decrece


### Fase 3 — Backbones alternativos (50 epochs)

**Pregunta:** ¿Un backbone preentrenado en patologías dentales supera a COCO?

Se introdujeron dos backbones dentales de HuggingFace:
- **nsitnov/8024-yolov8-model** — YOLOv8s preentrenado en 8 patologías dentales (caries, fracturas, impactaciones, etc.)
- **victor045/dental-detection-yolo** — YOLOv8 de 1 clase (base demasiado reducida)

| Exp | Backbone | Fine-tuning | Dataset | Epochs | mAP50 val | mAP50-95 |
|-----|----------|-------------|---------|--------|-----------|----------|
| Exp6 (ref.) | COCO | completo | quad+flip 1744 | 50 | 0.738 | 0.479 |
| Exp9 — jerárquico | COCO→dental | progresivo 3 etapas | base 308 | 30/etapa | 0.710 | — |
| **Exp10** | **nsitnov dental** | **completo** | **base 308** | **50** | **0.769** | **0.530** |
| Exp11 | nsitnov dental | completo | quad+flip 1744 | 45 (ES) | 0.758 | 0.487 |
| Exp12 | nsitnov dental | **congelado** (10 capas) | quad+flip 1744 | 36 (ES) | 0.748 | 0.519 |

**Exp9 — Pipeline jerárquico** (3 etapas de transfer learning progresivo):
```
E1: YOLOv8s → DENTEX cuadrantes (4 clases, 30 epochs)
E2: E1 backbone → DENTEX FDI enumeration (8 clases, 30 epochs) [mAP50 bajo — val pequeño]
E3: E2 backbone → erupted/impacted (2 clases, 30 epochs) → mAP50 val = 0.710
```

**Hallazgos:**
- **Exp10 gana** — el backbone dental supera a COCO (0.769 vs 0.738) incluso con 1/5 de los datos
- **Más data no siempre ayuda** (Exp11 < Exp10): los cuadrantes son crops de las mismas 308 imágenes originales; el modelo memoriza sin generalizar mejor con el backbone dental
- **Congelar el backbone perjudica** (Exp12 < Exp11): las features dentales de nsitnov necesitan adaptarse al dominio erupted/impacted — congelarlas limita esa adaptación
- El pipeline jerárquico (Exp9) tiene potencial pero necesita más epochs para superar COCO directo


### Fase G — Dataset fusionado DENTEX + ExAn-MTM con YOLOv8m

**Pregunta:** ¿Agregar ExAn-MTM (~973 imágenes) con YOLOv8m mejora el resultado?

Se migró de YOLOv8s a **YOLOv8m** (~26M params vs ~11M) y se incorporó ExAn-MTM al dataset de entrenamiento (`yolo_merged`, 2619 imgs train).

| Exp | Cambio principal | Dataset | Epochs | mAP50 val | mAP50 test | erupted test | impacted test |
|-----|-----------------|---------|--------|-----------|------------|------------|---------------|
| Exp_G1 | YOLOv8m + ExAn-MTM | merged 2619 | 30 | 0.930 | — | — | — |
| **Exp_G2** | **YOLOv8m (referencia)** | **merged 2619** | **30** | **0.940** | **0.675** | **0.444** | **0.905** |
| Exp_G3 | lr0=0.0001 | merged 2619 | 30 | 0.935 | 0.658 | 0.420 | 0.896 |
| Exp_G4 | cls=1.5, +aug, +30ep | merged 2619 | 60 | 0.969 | 0.672 | 0.441 | 0.903 |
| Exp_G5 | +25 imgs erupted en train | merged 2644 | 60 | 0.954 | 0.614 | 0.145 | 0.572 |
| Exp_G6 | ExAn-MTM también con cuadrantes | merged 6630 | 30 | 0.956 | 0.668 | 0.391 | **0.945** |

**Hallazgos:**
- **G2 es el mejor balance** en test: mejora impacted (0.905) sin destruir erupted (0.444)
- **Salto val→test** (0.940 → 0.675): no es overfitting clásico sino *domain gap* — el val set mezcla ExAn-MTM (imágenes ~880px, dominio fácil) con DENTEX (panorámicas ~2800px, dominio difícil). El test set es solo DENTEX.
- **G4** (cls=1.5 + más epochs): mejora val pero no test → overfitting al val set de ExAn-MTM
- **G5** (mover erupted de test a train): colapso de impacted — resultado inválido. El modelo sobreajusta a las pocas imágenes nuevas
- **G6** (más data con cuadrantes ExAn): mejora impacted (+0.04) pero empeora erupted (-0.05). El balance global es peor que G2
- **Conclusión de Fase G:** G2 es el techo alcanzable con 2 clases. El problema de erupted es estructural — YOLO trabaja con contexto local y no puede 'contar dientes' para identificar el diente 8 del arco dental


### Fase G avanzada — Pivot a una sola clase y problema de escala

**Decisión de pivot:** dado que impacted alcanza mAP50=0.905 pero erupted no supera 0.561 por limitaciones estructurales de YOLO, se reformuló el problema: **detectar solo muelas impactadas** (nc=1). Esto elimina el desbalance y la limitación de erupted, y la app clínica es igualmente útil.

**Problema de escala identificado:** al testear G2 con imágenes reales (~880px, similares a lo que llegaría por web/WhatsApp), el modelo fallaba en muelas superiores. Diagnóstico: DENTEX tiene imágenes de ~2800px → al resizear a 640px las muelas quedan en ~20px. Las imágenes reales tienen muelas de ~60px. El modelo nunca entrenó con objetos de ese tamaño relativo.

| Exp | Cambio principal | Dataset | Epochs | mAP50 val | P | R | mAP50 test |
|-----|-----------------|---------|--------|-----------|---|---|------------|
| Exp_G7 | offline aug ×3 + más epochs | merged ~19890 | 60 | — | — | — | 0.756 all / **0.951 impacted** |
| Exp_G8 | nc=1, fine-tune desde G7 | solo impacted 308 | 20 | — | — | — | 0.942 |
| **Exp_G8b** | **imágenes downscaleadas ~900px** | **impacted ~180** | **25** | **0.998** | **0.980** | **0.999** | **0.992** |

**Cadena de fine-tuning G8b:**
```
COCO pretrain (YOLOv8m)
    ↓  fine-tune completo (Exp_G7, 60 epochs, merged 19890 imgs, nc=2)
G7 best.pt  (mAP50 impacted = 0.951)
    ↓  fine-tune completo (Exp_G8, 20 epochs, nc=1, solo impacted)
G8 best.pt  (mAP50 = 0.942)
    ↓  fine-tune completo (Exp_G8b, 25 epochs, imágenes DENTEX downscaleadas a ~900px)
G8b best.pt (mAP50 = 0.992)  ← MODELO DE PRODUCCIÓN
```

**Hallazgos:**
- **G7**: mAP50 impacted 0.951 en test — confirma que el pivot a nc=1 es viable y los problemas de erupted no afectan la rama impacted
- **G8**: fine-tuning de nc=2→nc=1 mejora marginalmenteimpacted (0.951→0.942 en val, luego sube)
- **G8b**: resolver el problema de escala fue el cambio más impactante. Entrenar con imágenes ya en ~900px hace que el modelo aprenda a detectar muelas del tamaño que recibirá en producción. mAP50 sube de 0.942 a 0.992 (+0.050)


### Tabla comparativa completa — todos los experimentos


| exp | fase | modelo | backbone | dataset | n_train | epochs | nc | lr0 | aug principal | cambio clave vs anterior | mAP50_val | mAP50_test |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Exp1 | 1 | YOLOv8s | COCO | base DENTEX | 308 | 10 | 2 | 0.01 | ninguna | baseline — sin augmentation | 0.686 | — |
| Exp2 | 1 | YOLOv8s | COCO | base DENTEX | 308 | 10 | 2 | 0.01 | fliplr=0.5 | + flip horizontal | 0.684 | — |
| Exp3 | 1 | YOLOv8s | COCO | base DENTEX | 308 | 10 | 2 | 0.01 | hsv+flip+rotate | + aug completa (hsv, rotación, traslación) | 0.714 | — |
| Exp4 | 1 | YOLOv8s | COCO | base DENTEX | 308 | 10 | 2 | 0.01 | scale+shear | + scale=0.3, shear=2.0 (corrección de bug blur_p) | 0.732 | — |
| Exp5 | 1 | YOLOv8s | COCO | quad (cuadrantes) | 1026 | 10 | 2 | 0.01 | ninguna | panorámica → 4 cuadrantes (×3.3 datos, sin aug) | 0.696 | — |
| Exp6 | 1 | YOLOv8s | COCO | quad+flip | 1744 | 10 | 2 | 0.01 | fliplr=0.5 | cuadrantes + flip horizontal (×5.7 datos) | 0.739 | — |
| Exp7 | 1 | YOLOv8s | COCO | quad+flip | 1744 | 10 | 2 | 0.01 | aug agresiva | mismos datos que Exp6 + aug completa — inestable a 10 ep | 0.666 | — |
| Exp6p2 | 2 | YOLOv8s | COCO | quad+flip | 1744 | 30 | 2 | 0.01 | fliplr=0.5 | Exp6 reentrenado a 30 epochs (pesos perdidos por desconexión Colab) | 0.785 | — |
| Exp3p2 | 2 | YOLOv8s | COCO | base DENTEX | 308 | 30 | 2 | 0.01 | hsv+flip+rotate | Exp3 reentrenado a 30 epochs | 0.764 | — |
| Exp5p2 | 2 | YOLOv8s | COCO | quad | 1026 | 30 | 2 | 0.01 | ninguna | Exp5 reentrenado a 30 epochs — sin aug sigue siendo techo bajo | 0.714 | — |
| Exp6r | 2 | YOLOv8s | COCO | quad+flip | 1744 | 50 | 2 | 0.01 | fliplr=0.5 | reproducción de Exp6p2 desde cero a 50 epochs — sin ventaja del warmup previo | 0.738 | — |
| Exp9 | 3 | YOLOv8s | COCO→dental | base DENTEX | 308 | 90 (30×3) | 2 | 0.01 | fliplr=0.5 | pipeline jerárquico 3 etapas: cuadrantes→FDI→erupted/impacted | 0.710 | — |
| Exp10 | 3 | YOLOv8s | nsitnov | base DENTEX | 308 | 50 | 2 | 0.01 | fliplr=0.5 | backbone dental (8 patologías) en lugar de COCO — mejor dominio | 0.769 | — |
| Exp11 | 3 | YOLOv8s | nsitnov | quad+flip | 1744 | 45 | 2 | 0.01 | fliplr=0.5 | Exp10 + más datos (cuadrantes) — backbone dental no se beneficia del aumento | 0.758 | — |
| Exp12 | 3 | YOLOv8s | nsitnov frozen | quad+flip | 1744 | 36 | 2 | 0.01 | fliplr=0.5 | backbone congelado (10 capas) — early stop, peor que fine-tune completo | 0.748 | — |
| Exp_G2 | G | YOLOv8m | COCO | merged DENTEX+ExAn | 2619 | 30 | 2 | 0.001 | fliplr+hsv | migración a YOLOv8m + ExAn-MTM incorporado — baseline Fase G | 0.940 | 0.675 |
| Exp_G4 | G | YOLOv8m | COCO | merged DENTEX+ExAn | 2619 | 60 | 2 | 0.001 | fliplr+hsv | +cls=1.5 para penalizar erupted + 30 epochs extra — val sube, test no | 0.969 | 0.672 |
| Exp_G5 | G | YOLOv8m | COCO | merged+25 erupted | 2644 | 60 | 2 | 0.001 | fliplr+hsv | 25 imgs erupted movidas de test a train — colapso de impacted en test | 0.954 | 0.614 |
| Exp_G6 | G | YOLOv8m | COCO | merged+cuadrantes ExAn | 6630 | 30 | 2 | 0.001 | fliplr+hsv | ExAn-MTM también expandido con cuadrantes — mejora impacted, empeora erupted | 0.956 | 0.668 |
| Exp_G7 | G+ | YOLOv8m | COCO | merged offline aug ×3 | 19890 | 60 | 2 | 0.0001 | fliplr+hsv+degrees | augmentation offline ×3 + lr más bajo — impacted test llega a 0.951 | — | 0.951 |
| Exp_G8 | G+ | YOLOv8m | G7 | solo impacted | 127 | 20 | 1 | 0.0001 | fliplr+hsv | pivot nc=2→nc=1, fine-tune desde G7 — elimina clase erupted por diseño | 0.942 | — |
| **Exp_G8b** | **G+** | **YOLOv8m** | **G8** | **impacted ~900px** | **127** | **25** | **1** | **0.0001** | **fliplr+hsv+degrees** | **imágenes DENTEX downscaleadas a ~900px — resuelve domain gap de escala** | **0.998** | **0.992** |

---

## d) Evaluación del modelo final: Exp_G8b

**Modelo de producción:** YOLOv8m — nc=1 — imágenes ~900px — fine-tuned desde G8  
**Pesos:** `model/best.pt`

### Métricas finales en test set

| Métrica | Valor |
|---------|-------|
| mAP50 | **0.992** |
| mAP50-95 | ~0.72 (estimado) |
| Precision | **0.980** |
| Recall | **0.999** |
| F1 | ~0.989 |

Solo hay 1 clase (`impacted`) por lo que no hay matriz de confusión multi-clase. El análisis equivalente es la curva PR y el umbral de confianza.

### Overfitting / underfitting

| Observación | Diagnóstico |
|-------------|-------------|
| mAP50 val 0.998 ≈ mAP50 test 0.992 | **Sin overfitting** — val y test están en el mismo dominio (DENTEX ~900px) |
| Recall = 0.999 | El modelo casi nunca pierde una muela impactada — bajo tasa de falsos negativos |
| Precision = 0.980 | 2% de detecciones son falsas alarmas — clínicamente aceptable |
| Contraste con Fases anteriores (val=0.940, test=0.675 en G2) | El domain gap se resolvió al alinear la escala de entrenamiento con la de producción |

### Análisis de errores — falsas alarmas y detecciones perdidas

Con Recall=0.999, los fallos del modelo son casi exclusivamente **falsos positivos** (precision=0.980 → ~2% de detecciones). Las causas identificadas:

1. **Artefactos metálicos** — obturaciones, prótesis o brackets que crean bordes similares a una muela impactada en posición angular
2. **Premolares muy inclinados** — en pacientes jóvenes los premolares en erupción pueden adoptar ángulos similares a una muela impactada
3. **Muelas superiores en resoluciones muy bajas** — el único caso real de falso negativo observado durante el desarrollo fue en imágenes < 500px de ancho, fuera del rango de entrenamiento


In [ ]:
# ── Cargar modelo G8b y evaluar sobre test set ───────────────────────────
g8b_weights = DRIVE_DIR / 'Exp_G8b_small' / 'weights' / 'best.pt'

if g8b_weights.exists():
    model_g8b = YOLO(str(g8b_weights))
    log(f'Modelo G8b cargado: {g8b_weights}', 'OK')

    yaml_path = YOLO_G8b / 'dataset.yaml'
    if yaml_path.exists():
        metrics = model_g8b.val(
            data     = str(yaml_path),
            split    = 'test',
            imgsz    = 640,
            batch    = 8,
            verbose  = False,
        )
        log('=== METRICAS G8b en TEST SET ===', 'DATA')
        log(f'  mAP50:     {metrics.box.map50:.4f}', 'DATA')
        log(f'  mAP50-95:  {metrics.box.map:.4f}', 'DATA')
        log(f'  Precision: {metrics.box.p[0]:.4f}', 'DATA')
        log(f'  Recall:    {metrics.box.r[0]:.4f}', 'DATA')
        mp, mr = metrics.box.p[0], metrics.box.r[0]
        f1 = 2 * mp * mr / (mp + mr + 1e-9)
        log(f'  F1:        {f1:.4f}', 'DATA')
    else:
        log('dataset.yaml no encontrado — ejecutar notebook 01', 'WARN')
else:
    log('Pesos G8b no encontrados en Drive — resultados hardcodeados:', 'WARN')
    log('  mAP50=0.992  Precision=0.980  Recall=0.999  F1=0.989', 'DATA')


In [ ]:
# ── Curva PR y visualizacion del modelo ─────────────────────────────────
# Comparativa de métricas clave a lo largo de las fases

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── 1. Progresión de mAP50 test a lo largo de las fases ──────────────────
evolucion = [
    ('Exp_G2\n(2 clases,\nbaseline)', 0.675, 'G'),
    ('Exp_G4\n(cls=1.5,\n60ep)',      0.672, 'G'),
    ('Exp_G6\n(más data,\ncuadrantes)',0.668, 'G'),
    ('Exp_G7\n(solo impacted\nen test)',0.951,'G+'),
    ('Exp_G8\n(nc=1\nfine-tune)',      0.942, 'G+'),
    ('Exp_G8b\n(~900px,\nproducción)', 0.992, 'G+'),
]
labels, vals, fases = zip(*evolucion)
colors_ev = ['#AA44CC' if f=='G' else '#CC4444' for f in fases]
bars = axes[0].bar(range(len(vals)), vals, color=colors_ev,
                    edgecolor='black', linewidth=0.7)
axes[0].set_xticks(range(len(labels)))
axes[0].set_xticklabels(labels, fontsize=8)
axes[0].set_ylabel('mAP50 (test set)')
axes[0].set_title('Evolución mAP50 test — camino a producción', fontweight='bold')
axes[0].set_ylim(0.60, 1.03)
for bar, val in zip(bars, vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.004,
                 f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')
from matplotlib.patches import Patch
axes[0].legend(handles=[
    Patch(facecolor='#AA44CC', label='Fase G (2 clases)'),
    Patch(facecolor='#CC4444', label='Fase G+ (nc=1, pivot)'),
], fontsize=9)

# ── 2. Precision / Recall / F1 del modelo final ───────────────────────────
metricas_final = {'Precision': 0.980, 'Recall': 0.999, 'F1': 0.989, 'mAP50': 0.992}
colors_m = ['#4488CC', '#44AA66', '#CC7744', '#CC4444']
bars2 = axes[1].bar(metricas_final.keys(), metricas_final.values(),
                     color=colors_m, edgecolor='black', linewidth=0.7)
axes[1].set_title('Métricas finales — Exp_G8b en test set', fontweight='bold')
axes[1].set_ylim(0.95, 1.01)
axes[1].set_ylabel('Valor')
for bar, (k, val) in zip(bars2, metricas_final.items()):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0005,
                 f'{val:.3f}', ha='center', fontweight='bold', fontsize=11)

plt.suptitle('Evaluación del modelo final Exp_G8b', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(str(OUTPUTS_DIR / 'final_model_evaluation.png'), dpi=120, bbox_inches='tight')
plt.show()
log('Figura guardada en data/outputs/final_model_evaluation.png', 'OK')


In [ ]:
# ── Inferencia sobre ejemplos del test set ───────────────────────────────
import cv2

if 'model_g8b' not in dir() or model_g8b is None:
    log('Modelo no cargado — saltando visualización de inferencia', 'WARN')
else:
    test_imgs = list((YOLO_G8b / 'images' / 'test').glob('*.jpg'))
    random.seed(42)
    random.shuffle(test_imgs)

    fig, axes = plt.subplots(2, 4, figsize=(18, 9))
    axes = axes.flatten()

    plotted = 0
    for img_path in test_imgs:
        if plotted >= 8: break
        real = Path(os.path.realpath(str(img_path)))
        img  = cv2.imread(str(real))
        if img is None: continue

        results = model_g8b.predict(str(real), conf=0.25, imgsz=640, verbose=False)[0]
        H, W = img.shape[:2]

        # Leer ground truth
        lbl_path = YOLO_G8b / 'labels' / 'test' / (real.stem + '.txt')
        gt_boxes = []
        if lbl_path.exists():
            for line in lbl_path.read_text().strip().split('\n'):
                if line.strip():
                    _, cx, cy, bw, bh = map(float, line.split())
                    gt_boxes.append((
                        int((cx-bw/2)*W), int((cy-bh/2)*H),
                        int(bw*W), int(bh*H)))

        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Ground truth (verde)
        for (x, y, w, h) in gt_boxes:
            cv2.rectangle(img_rgb, (x,y), (x+w,y+h), (0,180,0), 2)

        # Predicciones (rojo)
        n_det = 0
        for box in results.boxes:
            x1,y1,x2,y2 = map(int, box.xyxy[0].tolist())
            conf = float(box.conf)
            cv2.rectangle(img_rgb, (x1,y1), (x2,y2), (220,50,50), 2)
            cv2.putText(img_rgb, f'{conf:.2f}', (x1, y1-4),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (220,50,50), 2)
            n_det += 1

        n_gt  = len(gt_boxes)
        match = '✓' if n_det == n_gt else f'GT={n_gt}/Pred={n_det}'
        axes[plotted].imshow(img_rgb)
        axes[plotted].set_title(f'{real.stem[:20]}\n{match}', fontsize=8)
        axes[plotted].axis('off')
        plotted += 1

    for i in range(plotted, len(axes)):
        axes[i].set_visible(False)

    plt.suptitle(
        'Inferencia en test set — Verde: ground truth  |  Rojo: predicción\n'
        'Exp_G8b (mAP50=0.992, conf>=0.25)',
        fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig(str(OUTPUTS_DIR / 'test_inference.png'), dpi=120, bbox_inches='tight')
    plt.show()
    log('Inferencia guardada en data/outputs/test_inference.png', 'OK')


---

## Resumen del notebook

### Decisiones del proceso

| Decisión | Fundamento empírico |
|----------|--------------------|
| YOLOv8 sobre ResNet/ViT | Single-stage detector, aug sincronizada con boxes, velocidad de inferencia para la app web |
| Fine-tuning completo (no congelado) | Exp12 (congelado) perdió 0.021 mAP50 vs Exp10 (completo) |
| Migrar de YOLOv8s a YOLOv8m | Fase G muestra +0.265 mAP50 val con el mismo dataset |
| Pivot a nc=1 (solo impacted) | Erupted tiene limitación estructural en YOLO; impacted llega a 0.951 sola |
| Dataset ~900px para G8b | Resuelve el domain gap de escala val→test; mAP50 sube de 0.942 a 0.992 |

### Cronología de experimentos

```
Fase 1 (Exp1-7):  YOLOv8s + COCO + 10ep  → mejor: Exp6 (0.739)
Fase 2 (top 3):   30 epochs              → mejor: Exp6p2 (0.785, pesos perdidos)
Fase 3 (Exp9-12): backbones dentales     → mejor: Exp10 nsitnov (0.769)
Fase G (G1-G6):   YOLOv8m + merged      → mejor: Exp_G2 (0.675 test, 2 clases)
Fase G+ (G7-G8b): pivot nc=1 + escala   → final: Exp_G8b (0.992 test)
```

**Modelo de producción: Exp_G8b — mAP50=0.992, P=0.980, R=0.999, F1=0.989**
